# CPSTN Draft6 Training

Implements the attached authoritative specification. Automatic GPKG/ZIP discovery, layer inspection, road-layer scoring, six-zone validation, static-vs-temporal traffic detection, sparse road graph construction, CPSTN training, smoke test, checkpoints, metrics, metadata, and results ZIP.

In [ ]:
from pathlib import Path
from collections import Counter
import os, re, json, zipfile, hashlib, random, sys, platform, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
SEED=42; random.seed(SEED); np.random.seed(SEED)
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import geopandas as gpd
import networkx as nx
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT=Path.cwd(); DATA_DIR=Path(os.getenv("CPSTN_DATA_DIR",ROOT/"data"))
OUT=Path(os.getenv("CPSTN_OUTPUT_DIR",ROOT/"outputs"))
for x in ["graph","processed","checkpoints","metrics","plots","metadata","training_results","testing_results"]:
    (OUT/x).mkdir(parents=True,exist_ok=True)
USE_SYNTHETIC_DEVELOPMENT_DATA=False
CFG={"history_steps":12,"horizon_steps":12,"hidden_channels":32,"dropout":.10,
     "temporal_dilations":[1,2,4],"learning_rate":.002,"weight_decay":1e-4,
     "gradient_clip":2.0,"max_epochs":50,"patience":8,"batch_size":8,"beta":.5,"seed":SEED}
ZONES={"south":"southern-zone-261002-free.gpkg","central":"central-zone-261002-free.gpkg",
"north":"northern-zone-261002-free.gpkg","east":"eastern-zone-261002-free.gpkg",
"west":"western-zone-261002-free.gpkg","north-east":"north-eastern-zone-261002-free.gpkg"}
CANDS={"road_id":["osm_id","osmid","road_id","edge_id","id","fid"],
"road_class":["fclass","highway","road_class","class","type"],
"road_name":["name","road_name","street","ref"],"ref":["ref","route","route_ref"],
"oneway":["oneway","one_way","direction"],"maxspeed":["maxspeed","max_speed","speed","speed_limit"],
"timestamp":["timestamp","datetime","date_time","date","time"],"speed":["speed","traffic_speed","observed_speed","measurement"],
"flow":["flow","volume","traffic_volume"],"occupancy":["occupancy"],"sensor_id":["sensor_id","sensor","detector_id","station_id"]}
POS=["osm_id","osmid","road_id","edge_id","fclass","highway","road_class","name","road_name","ref","oneway","maxspeed","bridge","tunnel","lanes","surface"]
NEG=["village","town","city","locality","place","building","landuse","amenity"]
def norm(x): return re.sub(r"[^a-z0-9]+","_",str(x).lower()).strip("_")
def cand(cols,names):
    d={norm(c):c for c in cols}
    return next((d[norm(x)] for x in names if norm(x) in d),None)
def js(p,x):
    def cv(v):
        if isinstance(v,(np.integer,np.floating)): return v.item()
        if isinstance(v,np.ndarray): return v.tolist()
        if isinstance(v,Path): return str(v)
        if isinstance(v,dict): return {str(k):cv(v) for k,v in v.items()}
        if isinstance(v,(list,tuple)): return [cv(a) for a in v]
        return v
    Path(p).write_text(json.dumps(cv(x),indent=2,default=str),encoding="utf8")
def sha(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

In [ ]:
def extract_zip_archives(root=DATA_DIR):
    root=Path(root); er=root/"_cpstn_extracted"; er.mkdir(exist_ok=True)
    report=[]
    for z in root.rglob("*.zip"):
        if er in z.parents: continue
        target=er/z.stem; target.mkdir(parents=True,exist_ok=True)
        try:
            with zipfile.ZipFile(z) as f:
                bad=f.testzip()
                if bad: raise RuntimeError(f"corrupt member: {bad}")
                f.extractall(target)
            report.append({"zip":str(z),"status":"ok","directory":str(target)})
        except Exception as e:
            report.append({"zip":str(z),"status":"error","error":str(e)})
    return report
def validate_gpkg(p):
    try:
        ls=gpd.list_layers(p)
        return {"path":str(p),"valid":len(ls)>0,"layers":ls.to_dict("records")}
    except Exception as e: return {"path":str(p),"valid":False,"error":str(e),"layers":[]}
def inspect_layer(p,layer):
    try:
        g=gpd.read_file(p,layer=layer)
        attrs=[c for c in g.columns if c!="geometry"]
        gt=g.geometry.geom_type.value_counts(dropna=False).to_dict()
        return {"layer":layer,"feature_count":len(g),"geometry_types":{str(k):int(v) for k,v in gt.items()},
                "crs":str(g.crs) if g.crs else None,"columns":list(map(str,g.columns)),
                "dtypes":{str(c):str(t) for c,t in g.dtypes.items()},
                "representative_attributes":g[attrs].head(3).replace({np.nan:None}).to_dict("records"),
                "missing_values":{str(c):int(g[c].isna().sum()) for c in attrs}}
    except Exception as e: return {"layer":layer,"feature_count":None,"geometry_types":{},"crs":None,"columns":[],"dtypes":{},"representative_attributes":[],"missing_values":{},"error":str(e)}
def score(info):
    n={norm(c) for c in info["columns"]}; gt=info["geometry_types"]
    lines=sum(v for k,v in gt.items() if "LineString" in k); points=sum(v for k,v in gt.items() if "Point" in k)
    polys=sum(v for k,v in gt.items() if "Polygon" in k); s=50 if lines else (-45 if points and not lines else -35 if polys and not lines else 0)
    ph=[x for x in POS if norm(x) in n]; nh=[x for x in NEG if norm(x) in n]
    s+=min(40,5*len(ph))-min(35,7*len(nh))
    why=(["contains LineString/MultiLineString geometry"] if lines else [])+([f"road fields: {', '.join(ph[:10])}"] if ph else [])+([f"negative fields: {', '.join(nh[:6])}"] if nh else [])
    if points and not lines: why+=["point-only geometry"]
    return s,why
def identify_road_layer(p):
    a=validate_gpkg(p)
    if not a["valid"]: return {"selected":None,"layers":[],"error":a.get("error","invalid GPKG")}
    ls=[]
    for r in a["layers"]:
        i=inspect_layer(p,r["name"]); s,w=score(i); i.update(road_score=s,selection_reasons=w); ls.append(i)
    lines=[x for x in ls if any("LineString" in k for k in x["geometry_types"])]
    best=max(lines,key=lambda x:x["road_score"],default=None)
    return {"selected":best if best and best["road_score"]>=20 else None,"layers":ls,
            "error":None if best and best["road_score"]>=20 else "No convincing road layer was found."}
def discover():
    if not DATA_DIR.exists(): raise FileNotFoundError(f"Data directory missing: {DATA_DIR}")
    z=extract_zip_archives(); ps=sorted(set(DATA_DIR.rglob("*.gpkg")))
    return [validate_gpkg(p) for p in ps],z
def zone_of(p):
    s=Path(p).stem.lower()
    for z,f in ZONES.items():
        if Path(f).stem.lower() in s or z.replace("-","") in re.sub("[^a-z]","",s): return z
    return None
def audit(p):
    d=identify_road_layer(p); a={"file":str(p),"file_size":Path(p).stat().st_size,"layers":d["layers"],
    "selected":d["selected"],"error":d["error"],"road_attributes":{},"traffic":False}
    if d["selected"]:
        c=d["selected"]["columns"]; a["road_attributes"]={k:cand(c,v) for k,v in CANDS.items()}
        a["traffic"]=bool(a["road_attributes"]["timestamp"] and a["road_attributes"]["speed"])
    return a
records,zips=discover(); audits=[audit(r["path"]) for r in records if r["valid"]]
print("Discovered GPKGs:",len(audits))
for a in audits:
    print("\n",Path(a["file"]).name)
    for l in a["layers"]:
        s,w=score(l); print(" ",l["layer"],l["geometry_types"],"score",s,"|","; ".join(w))
    print(" SELECTED:",(a["selected"] or {}).get("layer"),"|",a["error"])

In [ ]:
print("\n=== ALL LAYERS + ROAD-LAYER DECISIONS ===")
for a in audits:
    print("\n",Path(a["file"]).name)
    for l in a["layers"]:
        s,w=score(l);print(" ",l["layer"],"| geometry:",l["geometry_types"],"| score:",s,"|","; ".join(w))
    print(" SELECTED:",(a["selected"] or {}).get("layer"),"|",a["error"])
js(OUT/"metadata"/"dataset_audit.json",audits)

In [ ]:
print("=== DATASET AUDIT ===")
for a in audits:
    print(Path(a["file"]).name,"| road features:",(a["selected"] or {}).get("feature_count"),"| CRS:",(a["selected"] or {}).get("crs"),"| temporal:",a["traffic"])
print("ZIP extraction:",zips)

In [ ]:
byz={zone_of(a["file"]):a for a in audits if zone_of(a["file"])}
missing=[z for z in ZONES if z not in byz]
print("Expected zones:",list(ZONES));print("Detected zones:",sorted(byz));print("Missing:",missing)

In [ ]:
def endpoint(x): return (round(float(x[0]),7),round(float(x[1]),7))
def oneway(v):
    if pd.isna(v): return None
    s=str(v).lower().strip()
    return 1 if s in {"yes","true","1","forward"} else -1 if s in {"-1","reverse","backward"} else 0 if s in {"no","false","0","both","two-way","twoway"} else None
def num(v):
    m=re.search(r"-?\d+(?:\.\d+)?",str(v))
    return float(m.group()) if m else None
def build_graph(a,limit=None):
    s=a["selected"]; gdf=gpd.read_file(a["file"],layer=s["layer"])
    invalid=int((~gdf.geometry.is_valid).sum()) if len(gdf) else 0
    gdf=gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf=gdf[gdf.geometry.geom_type.isin(["LineString","MultiLineString"])].copy()
    gdf["geometry"]=gdf.geometry.make_valid(); gdf=gdf.explode(index_parts=False,ignore_index=True)
    gdf=gdf[gdf.geometry.geom_type=="LineString"].drop_duplicates("geometry")
    m={k:cand(gdf.columns,v) for k,v in CANDS.items()}; G=nx.MultiDiGraph()
    for i,r in gdf.iterrows():
        q=r.geometry
        if len(q.coords)<2: continue
        u,v=endpoint(q.coords[0]),endpoint(q.coords[-1])
        if u==v: continue
        d={"edge_index":int(i),"length":float(q.length),"geometry_wkt":q.wkt,
           "road_id":str(r[m["road_id"]]) if m["road_id"] else str(i),
           "road_class":str(r[m["road_class"]]) if m["road_class"] else None,
           "road_name":str(r[m["road_name"]]) if m["road_name"] else None,
           "ref":str(r[m["ref"]]) if m["ref"] else None,
           "oneway":oneway(r[m["oneway"]]) if m["oneway"] else None,
           "maxspeed":num(r[m["maxspeed"]]) if m["maxspeed"] else None}
        G.add_edge(u,v,key=int(i),**d)
        if d["oneway"]!=1: G.add_edge(v,u,key=(int(i),"rev"),**d)
    U=nx.Graph(G); deg=[d for _,d in U.degree()]; comps=list(nx.connected_components(U))
    st={"nodes":G.number_of_nodes(),"edges":G.number_of_edges(),"connected_components":len(comps),
        "largest_connected_component":max(map(len,comps),default=0),"isolated_nodes":sum(d==0 for d in deg),
        "degree_mean":float(np.mean(deg)) if deg else 0,"degree_median":float(np.median(deg)) if deg else 0,
        "invalid_geometry_count":invalid,"maxspeed_available_fraction":float(np.mean([d["maxspeed"] is not None for *_,d in G.edges(data=True)])) if G.edges else 0}
    return G,st,m
def edge_index(G):
    ns=list(G.nodes); ix={n:i for i,n in enumerate(ns)}; e=[(ix[u],ix[v]) for u,v in G.edges if u in ix and v in ix and u!=v]
    return torch.tensor(e or [(i,i) for i in range(len(ns))],dtype=torch.long).t().contiguous(),ns

In [ ]:
rep=next((a for a in audits if a["selected"]),None)
if rep:
    G,st,map_=build_graph(rep);E,nodes=edge_index(G)
    print("Graph stats:",json.dumps(st,indent=2))
else: print("No real road layer available; graph construction cannot proceed.")

In [ ]:
def traffic_info(a):
    s=a["selected"]; g=gpd.read_file(a["file"],layer=s["layer"])
    t=cand(g.columns,CANDS["timestamp"]); sp=cand(g.columns,CANDS["speed"])
    if not t or not sp: return {"available":False,"timestamp":t,"speed":sp,"message":"The supplied datasets contain road-network topology and attributes but do not contain temporal traffic observations."}
    ts=pd.to_datetime(g[t],errors="coerce",utc=True); return {"available":ts.nunique(dropna=True)>=2,"timestamp":t,"speed":sp,"unique_timestamps":int(ts.nunique(dropna=True))}
class Scaler:
    def fit(self,x):
        self.mean=float(np.mean(x)); self.std=float(np.std(x)); self.std=max(self.std,1e-6); return self
    def transform(self,x): return (np.asarray(x,dtype=np.float32)-self.mean)/self.std
    def inverse(self,x): return np.asarray(x)*self.std+self.mean
    def state(self): return {"mean":self.mean,"std":self.std}
class GraphConv(nn.Module):
    def __init__(self,c): super().__init__(); self.lin=nn.Linear(c,c)
    def forward(self,x,e):
        # x=B,T,N,C; sparse edge aggregation, never dense NxN.
        src,dst=e; a=torch.zeros_like(x); a.index_add_(2,dst,x[:,:,src,:])
        d=torch.zeros(x.shape[2],device=x.device); d.index_add_(0,dst,torch.ones_like(dst,dtype=x.dtype))
        return self.lin((a+x)/(d.clamp_min(1).view(1,1,-1,1)+1))
class CPSTN(nn.Module):
    def __init__(self,nodes,h=32,horizon=12,dils=[1,2,4],drop=.1,e=None):
        super().__init__(); self.p=nn.Linear(1,h); self.g=GraphConv(h); self.gate=nn.Conv1d(h,2*h,1)
        self.ts=nn.ModuleList([nn.Conv2d(h,h,(3,1),dilation=(d,1),padding=((3-1)*d,0)) for d in dils])
        self.drop=nn.Dropout(drop); self.r=nn.Linear(h,h); self.head=nn.Linear(h,horizon); self.register_buffer("edge_index",e); self.nodes=nodes
    def forward(self,x):
        h=self.g(self.p(x),self.edge_index); z=h.permute(0,3,1,2); q,k=self.gate(z).chunk(2,1); z=torch.tanh(q)*torch.sigmoid(k)
        for c in self.ts: z=self.drop(torch.relu(c(z)[:,:,:z.shape[2],:]))+z
        z=z.permute(0,2,3,1); return self.head(z[:,-1]+self.r(h[:,-1]))
def windows(df,H=12,K=12):
    p=df.pivot_table(index="timestamp",columns="road_id",values="speed",aggfunc="mean").sort_index()
    A=p.to_numpy(np.float32); X=[];Y=[]
    for i in range(max(0,len(A)-H-K+1)): X.append(A[i:i+H]);Y.append(A[i+H:i+H+K])
    return np.array(X),np.array(Y),list(map(str,p.columns))
def temporal_zone(a):
    s=a["selected"]; g=gpd.read_file(a["file"],layer=s["layer"])
    t=cand(g.columns,CANDS["timestamp"]); sp=cand(g.columns,CANDS["speed"]); rid=cand(g.columns,CANDS["road_id"])
    if not (t and sp and rid):
        raise RuntimeError(f"{zone_of(a['file'])}: temporal speed cannot be mapped to a stable road/segment ID.")
    return pd.DataFrame({"road_id":g[rid].astype(str),"timestamp":pd.to_datetime(g[t],errors="coerce",utc=True),"speed":pd.to_numeric(g[sp],errors="coerce")}).dropna().sort_values("timestamp")
def validate_traffic_to_graph(a,G):
    rid=a["road_attributes"].get("road_id")
    if not rid:
        raise RuntimeError(f"{zone_of(a['file'])}: no stable road ID for traffic-to-graph mapping.")
    graph_ids={str(d.get("road_id")) for *_,d in G.edges(data=True) if d.get("road_id") is not None}
    g=gpd.read_file(a["file"],layer=a["selected"]["layer"])
    observed=set(g[rid].dropna().astype(str))
    overlap=len(observed & graph_ids)
    if not overlap:
        raise RuntimeError(
            f"{zone_of(a['file'])}: temporal traffic road IDs do not match graph edge IDs. "
            "Probable cause: traffic observations and road geometry use different identifiers. "
            "How to fix it: provide a stable shared segment ID or a documented spatial mapping."
        )
    return overlap/len(observed) if observed else 0.0
def split(X,Y):
    n=len(X); a=max(1,int(.7*n)); b=min(n-1,max(a+1,int(.85*n))); return (X[:a],Y[:a]),(X[a:b],Y[a:b]),(X[b:],Y[b:])
def score_metrics(y,p):
    e=p-y; mae=np.mean(np.abs(e)); rmse=np.sqrt(np.mean(e**2)); mape=np.mean(np.abs(e)/np.maximum(np.abs(y),1e-6))*100
    ss=np.sum((y-y.mean())**2); r2=1-np.sum(e**2)/ss if ss>1e-12 else np.nan
    return {"MAE":float(mae),"RMSE":float(rmse),"MAPE":float(mape),"R2":float(r2)}
def persist(X,K): return np.repeat(X[:,-1,:,None],K,axis=3)[:,:,:,0]

In [ ]:
ti={zone_of(a["file"]):traffic_info(a) for a in audits if a["selected"]};display(pd.DataFrame(ti).T)
for z,v in ti.items():
    if not v["available"]: print(z,":",v["message"])
print("Synthetic development data:",USE_SYNTHETIC_DEVELOPMENT_DATA)

## 15–18. Preprocessing, chronological split, and window generation

Traffic observations must be timestamped and mapped to stable road IDs. Scalers are fit only on training observations.

In [ ]:
print("Training gate follows below; no partial six-zone or static-only training is permitted.")

In [ ]:
# Full six-zone gate: never silently train on a partial collection.
byz={zone_of(a["file"]):a for a in audits if zone_of(a["file"])}
missing=[z for z in ZONES if z not in byz]
if missing: raise RuntimeError("Training cannot start.\nMissing Indian regional dataset(s): "+", ".join(missing))
if any(not traffic_info(a)["available"] for a in byz.values()):
    raise RuntimeError("Training cannot start. Real temporal traffic observations were not detected for every required zone. Static maxspeed is not observed traffic.")
traffic={z:traffic_info(a) for z,a in byz.items()}; js(OUT/"metadata"/"traffic_audit.json",traffic)
print("Six-zone validation passed.")
# Build representative real road graph.
rep=next(iter(byz.values())); G,graph_stats,mappings=build_graph(rep)
E,node_ids=edge_index(G); js(OUT/"graph"/"representative_stats.json",graph_stats)
pd.DataFrame([{"u":str(u),"v":str(v),"length":d.get("length"),"road_id":d.get("road_id"),"road_class":d.get("road_class"),"road_name":d.get("road_name"),"oneway":d.get("oneway"),"maxspeed":d.get("maxspeed")} for u,v,d in G.edges(data=True)]).to_parquet(OUT/"graph"/"representative_edges.parquet",index=False)

# Real temporal data only. Chronological split and scaler fitted only on training data.
parts={}
for z,a in byz.items():
    X,Y,nodes=windows(temporal_zone(a),CFG["history_steps"],CFG["horizon_steps"]); tr,va,te=split(X,Y)
    parts[z]={"tr":tr,"va":va,"te":te,"nodes":nodes}
common=set.intersection(*(set(v["nodes"]) for v in parts.values()))
if not common: raise RuntimeError("No common road IDs across zones; cannot safely align a combined model.")
order=sorted(common); N=len(order)
def align(pair,nodes):
    ix=[nodes.index(n) for n in order]; return pair[0][:,:,ix],pair[1][:,:,ix]
trX,trY=zip(*(align(v["tr"],v["nodes"]) for v in parts.values())); vaX,vaY=zip(*(align(v["va"],v["nodes"]) for v in parts.values()))
trX=np.concatenate(trX);trY=np.concatenate(trY);vaX=np.concatenate(vaX);vaY=np.concatenate(vaY)
sc=Scaler().fit(trX.reshape(-1)); trX=sc.transform(trX)[...,None]; vaX=sc.transform(vaX)[...,None]
# Targets are B,N,H.
trY=sc.transform(trY).transpose(0,2,1); vaY=sc.transform(vaY).transpose(0,2,1)
# Graph must be compatible with the exact traffic node ordering.
if len(node_ids)!=N: raise RuntimeError(f"Graph/traffic node mismatch: graph={len(node_ids)}, traffic={N}. Align IDs before training.")
model=CPSTN(N,CFG["hidden_channels"],CFG["horizon_steps"],CFG["temporal_dilations"],CFG["dropout"],E).to(DEVICE)
lossfn=nn.SmoothL1Loss(beta=CFG["beta"]); opt=torch.optim.AdamW(model.parameters(),lr=CFG["learning_rate"],weight_decay=CFG["weight_decay"])
loader=DataLoader(list(zip(torch.tensor(trX,dtype=torch.float32),torch.tensor(trY,dtype=torch.float32))),batch_size=CFG["batch_size"],shuffle=True)
vloader=DataLoader(list(zip(torch.tensor(vaX,dtype=torch.float32),torch.tensor(vaY,dtype=torch.float32))),batch_size=CFG["batch_size"])
# Mandatory smoke test.
xb,yb=next(iter(loader)); xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); pp=model(xb); sl=lossfn(pp,yb)
if not torch.isfinite(sl): raise RuntimeError("Smoke test failed: non-finite loss.")
sl.backward()
if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()): raise RuntimeError("Smoke test failed: non-finite gradients.")
torch.nn.utils.clip_grad_norm_(model.parameters(),CFG["gradient_clip"]);opt.step();print("SMOKE TEST PASSED:",float(sl))

best=float("inf");best_state=None;history=[];wait=0
for ep in range(1,CFG["max_epochs"]+1):
    model.train(); tl=[]
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE);opt.zero_grad();p=model(xb);l=lossfn(p,yb)
        if not torch.isfinite(l): raise RuntimeError(f"NaN/Inf loss at epoch {ep}.")
        l.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG["gradient_clip"]);opt.step();tl.append(float(l))
    model.eval();vl=[]
    with torch.no_grad():
        for xb,yb in vloader: vl.append(float(lossfn(model(xb.to(DEVICE)),yb.to(DEVICE))))
    tv, vv=float(np.mean(tl)),float(np.mean(vl));history.append({"epoch":ep,"train_loss":tv,"val_loss":vv})
    if vv<best: best=vv;wait=0;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else: wait+=1
    if wait>=CFG["patience"]: break
model.load_state_dict(best_state)
hist=pd.DataFrame(history);hist.to_csv(OUT/"metrics"/"training_history.csv",index=False)
meta={"configuration":CFG,"node_order":order,"graph_stats":graph_stats,"mappings":mappings,"edge_index":E.cpu().tolist(),"graph_node_ids":[str(n) for n in nodes],"scaler":sc.state(),
      "dataset_metadata":{z:{"file":a["file"],"sha256":sha(a["file"]),"layer":a["selected"]["layer"]} for z,a in byz.items()},
      "feature_configuration":{"input_features":1,"target":"speed"}}
ck={"model_state_dict":model.state_dict(),"optimizer_state":opt.state_dict(),"epoch":history[-1]["epoch"],"best_validation_loss":best,
    "configuration":{**CFG,"node_count":N,"input_features":1},"node_order":order,"graph_metadata":meta,
    "scaler_parameters":sc.state(),"dataset_metadata":meta["dataset_metadata"],"feature_configuration":meta["feature_configuration"]}
torch.save(ck,OUT/"checkpoints"/"CPSTN-Draft6_best.pt");torch.save(ck,OUT/"checkpoints"/"CPSTN-Draft6_final.pt")
js(OUT/"metadata"/"training_summary.json",{"best_validation_loss":best,"epochs":len(history),"device":str(DEVICE),"zones":list(byz),"synthetic":False})
try:
    import matplotlib.pyplot as plt
    plt.figure();plt.plot(hist.epoch,hist.train_loss,label="train");plt.plot(hist.epoch,hist.val_loss,label="validation");plt.legend();plt.tight_layout();plt.savefig(OUT/"plots"/"training_curve.png");plt.close()
except Exception: pass
# ZIP excludes original GPKGs.
with zipfile.ZipFile(OUT/"CPSTN_training_draft6_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for root in [OUT/"graph",OUT/"processed",OUT/"checkpoints",OUT/"metrics",OUT/"plots",OUT/"metadata",OUT/"training_results"]:
        for f in root.rglob("*"):
            if f.is_file(): z.write(f,f.relative_to(OUT))
print("TRAINING COMPLETE:",OUT/"CPSTN_training_draft6_results.zip")

## Final Training Summary / Routing Compatibility

The generated graph artifacts retain edge/node identity, geometry-derived length, speed/maxspeed, one-way information, and metadata needed for downstream predicted-speed → travel-time → dynamic-cost → A* routing. No routing result is fabricated when traffic predictions are unavailable.